In [ ]:
import pandas as pd

food_candidates = pd.read_csv("../preprocessed/final_food_candidates_dm_ht.csv")


In [ ]:
patient_target = {
    "diet_type": "DM_HT_OBESITY",
    "energy_kcal": 1600,
    "carb_g": 240,
    "protein_g": 60,
    "fat_g": 44,
    "sodium_mg_max": 2000,
    "fiber_g_min": 25
}

In [ ]:
def filter_foods_for_disease(df, diet_type):
    filtered = df.copy()

    # DM: remove sugar category if exists
    if "DM" in diet_type:
        filtered = filtered[filtered["category_code"] != "G"]

    # Hypertension: remove very high sodium foods
    if "HT" in diet_type:
        filtered = filtered[
            filtered["sodium_mg_per_portion"].isna() |
            (filtered["sodium_mg_per_portion"] <= 400)
        ]

    # Obesity: remove very high energy per portion
    if "OBESITY" in diet_type:
        filtered = filtered[
            filtered["energy_kcal_per_portion"] <= 250
        ]

    return filtered


candidate_for_patient = filter_foods_for_disease(
    food_candidates,
    patient_target["diet_type"]
)

print("Candidate foods after disease filter:", len(candidate_for_patient))

display(
    candidate_for_patient[[
        "food_code",
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]].head(20)
)

In [ ]:
portion_plan = {
    1100: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 0, "M": 3, "G": 1},
    1200: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 1, "M": 3, "G": 2},
    1300: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 2, "SS": 1, "M": 4, "G": 2},
    1400: {"MP": 3, "LH": 3, "LN": 3, "S": 2, "B": 2, "SS": 0, "M": 3, "G": 3},
    1500: {"MP": 3, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 0, "M": 4, "G": 3},
    1600: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 2, "SS": 0, "M": 4, "G": 2},
    1700: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 3, "G": 2},
    1800: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    1900: {"MP": 4, "LH": 4, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    2000: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2100: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2200: {"MP": 4, "LH": 3, "LN": 3, "S": 4, "B": 4, "SS": 2, "M": 5, "G": 4},
    2300: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 0, "M": 5, "G": 4},
    2400: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
    2500: {"MP": 5.5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
}

In [ ]:
def get_disease_constraints(diet_type, energy_kcal):
    """
    diet_type options:
    HT
    HT_OBESITY
    DM
    DM_OBESITY
    DM_HT
    DM_HT_OBESITY
    """

    constraints = {
        "energy_target": energy_kcal,
        "energy_mode": "maintenance",
        "carb_min_g": None,
        "carb_max_g": None,
        "fat_min_g": None,
        "fat_max_g": None,
        "sodium_max_mg": None,
        "potassium_min_mg": None,
        "calcium_min_mg": None,
        "fiber_min_g": None,
        "vegetable_fruit_min_portions": None,
        "sugar_max_portions": None
    }

    # 1. Hipertensi tanpa obesitas
    if diet_type == "HT":
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 2. Hipertensi dengan obesitas
    elif diet_type == "HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = energy_kcal * 0.55 / 4
        constraints["carb_max_g"] = energy_kcal * 0.65 / 4
        constraints["fat_min_g"] = energy_kcal * 0.20 / 9
        constraints["fat_max_g"] = energy_kcal * 0.25 / 9
        constraints["sodium_max_mg"] = 2300
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 3. DM tanpa obesitas
    elif diet_type == "DM":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sugar_max_portions"] = 0

    # 4. DM dengan obesitas
    elif diet_type == "DM_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 5. DM + Hipertensi tanpa obesitas
    elif diet_type == "DM_HT":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2300
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 6. DM + Hipertensi + Obesitas
    elif diet_type == "DM_HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = 130
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2000
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    return constraints

In [ ]:
constraints = get_disease_constraints("DM_HT_OBESITY", 1600)
constraints

In [ ]:
def adjust_portion_plan_for_disease(base_plan, diet_type):
    plan = base_plan.copy()

    # DM: avoid sugar portions
    if "DM" in diet_type:
        plan["G"] = 0

    # Obesity: also reduce oil/sugar where possible
    if "OBESITY" in diet_type:
        plan["G"] = 0
        if "M" in plan:
            plan["M"] = max(0, plan["M"] - 1)

    # DM/HT: prioritize vegetables and fruits
    if diet_type in ["DM_OBESITY", "DM_HT", "DM_HT_OBESITY"]:
        current_s_b = plan.get("S", 0) + plan.get("B", 0)
        if current_s_b < 5:
            plan["S"] = plan.get("S", 0) + (5 - current_s_b)

    return plan

In [ ]:
base_plan = portion_plan[1600]
disease_plan = adjust_portion_plan_for_disease(base_plan, "DM_HT_OBESITY")

print(disease_plan)

In [ ]:
# ============================================================
# 3. Pick nearest available energy level
# ============================================================

def get_nearest_energy_level(energy_kcal, available_levels):
    return min(available_levels, key=lambda x: abs(x - energy_kcal))


def adjust_portion_plan_for_disease(base_plan, diet_type):
    plan = base_plan.copy()

    # DM: sugar should be avoided or minimized
    if "DM" in diet_type:
        plan["G"] = 0

    # Obesity: reduce sugar and oil portions
    if "OBESITY" in diet_type:
        plan["G"] = 0
        if "M" in plan:
            plan["M"] = max(0, plan["M"] - 1)

    # DM obesity / DM+HT: ensure vegetable + fruit >= 5 portions
    if diet_type in ["DM_OBESITY", "DM_HT", "DM_HT_OBESITY"]:
        current_sb = plan.get("S", 0) + plan.get("B", 0)
        if current_sb < 5:
            plan["S"] = plan.get("S", 0) + (5 - current_sb)

    return plan

In [ ]:
# ============================================================
# 4. Disease-aware food filter
# ============================================================

def filter_foods_for_disease(df, diet_type):
    filtered = df.copy()

    # DM: remove sugar category
    if "DM" in diet_type and "category_code" in filtered.columns:
        filtered = filtered[filtered["category_code"] != "G"]

    # Hypertension: remove high-sodium single portions
    if "HT" in diet_type and "sodium_mg_per_portion" in filtered.columns:
        filtered = filtered[
            filtered["sodium_mg_per_portion"].isna() |
            (filtered["sodium_mg_per_portion"] <= 400)
        ]

    # Obesity: avoid very high energy single portions
    if "OBESITY" in diet_type and "energy_kcal_per_portion" in filtered.columns:
        filtered = filtered[
            filtered["energy_kcal_per_portion"] <= 250
        ]

    return filtered

In [ ]:
# ============================================================
# 5. Example patient target
# ============================================================

diet_type = "DM_HT_OBESITY"
energy_target = 1600

constraints = get_disease_constraints(diet_type, energy_target)

nearest_energy = get_nearest_energy_level(
    energy_target,
    list(portion_plan.keys())
)

base_plan = portion_plan[nearest_energy]
adjusted_plan = adjust_portion_plan_for_disease(base_plan, diet_type)

candidate_for_patient = filter_foods_for_disease(food_candidates, diet_type)

print("Diet type:", diet_type)
print("Energy target:", energy_target)
print("Nearest portion plan:", nearest_energy)
print("\nDisease constraints:")
print(constraints)

print("\nBase portion plan:")
print(base_plan)

print("\nAdjusted portion plan:")
print(adjusted_plan)

print("\nTotal food candidates before filter:", len(food_candidates))
print("Total food candidates after disease filter:", len(candidate_for_patient))

category_after_filter = (
    candidate_for_patient
    .groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(category_after_filter)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 6. Rule-based menu generator
# ============================================================

def generate_rule_based_menu(candidate_df, adjusted_plan, random_state=42):
    np.random.seed(random_state)

    selected_rows = []

    for category, n_portions in adjusted_plan.items():
        n_portions_int = int(np.floor(n_portions))

        if n_portions_int <= 0:
            continue

        category_foods = candidate_df[
            candidate_df["category_code"] == category
        ].copy()

        if category_foods.empty:
            print(f"Warning: No foods available for category {category}")
            continue

        # Select with replacement if available foods are fewer than required portions
        replace_status = len(category_foods) < n_portions_int

        chosen = category_foods.sample(
            n=n_portions_int,
            replace=replace_status,
            random_state=random_state
        )

        chosen["selected_category"] = category
        chosen["selected_portion"] = 1

        selected_rows.append(chosen)

    if len(selected_rows) == 0:
        return pd.DataFrame()

    menu = pd.concat(selected_rows, ignore_index=True)

    return menu

In [ ]:
def add_food_score_for_dm_ht_obesity(df):
    df = df.copy()

    # Avoid division errors
    df["fiber_density"] = df["fiber_g_per_portion"] / df["energy_kcal_per_portion"].replace(0, np.nan)
    df["sodium_density"] = df["sodium_mg_per_portion"] / df["energy_kcal_per_portion"].replace(0, np.nan)

    # Fill missing values
    df["fiber_density"] = df["fiber_density"].fillna(0)
    df["sodium_density"] = df["sodium_density"].fillna(df["sodium_density"].max())

    # Higher score is better
    df["food_score"] = (
        df["fiber_density"] * 100
        - df["sodium_density"] * 10
        + df["protein_g_per_portion"].fillna(0) * 0.5
    )

    return df

In [ ]:
def generate_score_based_menu(candidate_df, adjusted_plan):
    selected_rows = []

    scored_df = add_food_score_for_dm_ht_obesity(candidate_df)

    for category, n_portions in adjusted_plan.items():
        n_portions_int = int(np.floor(n_portions))

        if n_portions_int <= 0:
            continue

        category_foods = scored_df[
            scored_df["category_code"] == category
        ].copy()

        if category_foods.empty:
            print(f"Warning: No foods available for category {category}")
            continue

        # Select highest scored foods in each category
        chosen = category_foods.sort_values(
            "food_score",
            ascending=False
        ).head(n_portions_int)

        # If not enough foods, repeat best foods
        if len(chosen) < n_portions_int:
            extra = category_foods.sort_values(
                "food_score",
                ascending=False
            ).head(1)

            while len(chosen) < n_portions_int:
                chosen = pd.concat([chosen, extra], ignore_index=True)

        chosen["selected_category"] = category
        chosen["selected_portion"] = 1

        selected_rows.append(chosen)

    if len(selected_rows) == 0:
        return pd.DataFrame()

    return pd.concat(selected_rows, ignore_index=True)

In [ ]:
daily_menu = generate_score_based_menu(
    candidate_for_patient,
    adjusted_plan
)

nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

existing_nutrient_cols = [
    col for col in nutrient_cols if col in daily_menu.columns
]

daily_totals = daily_menu[existing_nutrient_cols].sum(numeric_only=True)
menu_evaluation = evaluate_menu(daily_totals, constraints)

display(
    daily_menu[[
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion",
        "food_score"
    ]]
)

display(daily_totals.reset_index().rename(columns={"index": "nutrient", 0: "total"}))
display(menu_evaluation)

In [ ]:
daily_menu = generate_rule_based_menu(
    candidate_for_patient,
    adjusted_plan,
    random_state=42
)

display(
    daily_menu[[
        "food_code",
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "protein_g_per_portion",
        "fat_g_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]]
)

In [ ]:
nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

existing_nutrient_cols = [
    col for col in nutrient_cols if col in daily_menu_improved.columns
]

# Force numeric again after pd.concat
for col in existing_nutrient_cols:
    daily_menu_improved[col] = pd.to_numeric(
        daily_menu_improved[col],
        errors="coerce"
    )

daily_totals_improved = daily_menu_improved[existing_nutrient_cols].sum()

menu_evaluation_improved = evaluate_menu(
    daily_totals_improved,
    constraints
)

display(daily_totals_improved.reset_index().rename(columns={"index": "nutrient", 0: "total"}))
display(menu_evaluation_improved)

In [ ]:
# ============================================================
# 8. Evaluate menu against constraints
# ============================================================

def evaluate_menu(daily_totals, constraints):
    evaluation = []

    energy_total = daily_totals.get("energy_kcal_per_portion", np.nan)
    carb_total = daily_totals.get("carb_g_per_portion", np.nan)
    protein_total = daily_totals.get("protein_g_per_portion", np.nan)
    fat_total = daily_totals.get("fat_g_per_portion", np.nan)
    sodium_total = daily_totals.get("sodium_mg_per_portion", np.nan)
    potassium_total = daily_totals.get("potassium_mg_per_portion", np.nan)
    calcium_total = daily_totals.get("calcium_mg_per_portion", np.nan)
    fiber_total = daily_totals.get("fiber_g_per_portion", np.nan)

    # Energy: allow ±10% for rule-based prototype
    energy_target = constraints["energy_target"]
    energy_min = energy_target * 0.90
    energy_max = energy_target * 1.10

    evaluation.append({
        "constraint": "energy",
        "value": energy_total,
        "target_or_limit": f"{energy_min:.1f} - {energy_max:.1f}",
        "status": "pass" if energy_min <= energy_total <= energy_max else "not_pass"
    })

    # Carbohydrate minimum
    if constraints.get("carb_min_g") is not None:
        evaluation.append({
            "constraint": "carb_min",
            "value": carb_total,
            "target_or_limit": f">= {constraints['carb_min_g']:.1f}",
            "status": "pass" if carb_total >= constraints["carb_min_g"] else "not_pass"
        })

    # Carbohydrate maximum
    if constraints.get("carb_max_g") is not None:
        evaluation.append({
            "constraint": "carb_max",
            "value": carb_total,
            "target_or_limit": f"<= {constraints['carb_max_g']:.1f}",
            "status": "pass" if carb_total <= constraints["carb_max_g"] else "not_pass"
        })

    # Fat minimum
    if constraints.get("fat_min_g") is not None:
        evaluation.append({
            "constraint": "fat_min",
            "value": fat_total,
            "target_or_limit": f">= {constraints['fat_min_g']:.1f}",
            "status": "pass" if fat_total >= constraints["fat_min_g"] else "not_pass"
        })

    # Fat maximum
    if constraints.get("fat_max_g") is not None:
        evaluation.append({
            "constraint": "fat_max",
            "value": fat_total,
            "target_or_limit": f"<= {constraints['fat_max_g']:.1f}",
            "status": "pass" if fat_total <= constraints["fat_max_g"] else "not_pass"
        })

    # Sodium maximum
    if constraints.get("sodium_max_mg") is not None:
        evaluation.append({
            "constraint": "sodium_max",
            "value": sodium_total,
            "target_or_limit": f"<= {constraints['sodium_max_mg']:.1f}",
            "status": "pass" if sodium_total <= constraints["sodium_max_mg"] else "not_pass"
        })

    # Fiber minimum
    if constraints.get("fiber_min_g") is not None:
        evaluation.append({
            "constraint": "fiber_min",
            "value": fiber_total,
            "target_or_limit": f">= {constraints['fiber_min_g']:.1f}",
            "status": "pass" if fiber_total >= constraints["fiber_min_g"] else "not_pass"
        })

    # Potassium minimum
    if constraints.get("potassium_min_mg") is not None:
        evaluation.append({
            "constraint": "potassium_min",
            "value": potassium_total,
            "target_or_limit": f">= {constraints['potassium_min_mg']:.1f}",
            "status": "pass" if potassium_total >= constraints["potassium_min_mg"] else "not_pass"
        })

    # Calcium minimum
    if constraints.get("calcium_min_mg") is not None:
        evaluation.append({
            "constraint": "calcium_min",
            "value": calcium_total,
            "target_or_limit": f">= {constraints['calcium_min_mg']:.1f}",
            "status": "pass" if calcium_total >= constraints["calcium_min_mg"] else "not_pass"
        })

    return pd.DataFrame(evaluation)


menu_evaluation = evaluate_menu(daily_totals, constraints)

display(menu_evaluation)

In [ ]:
# ============================================================
# 9. Check selected category distribution
# ============================================================

selected_category_count = (
    daily_menu
    .groupby("category_code")
    .size()
    .reset_index(name="selected_portions")
)

display(selected_category_count)

print("Adjusted portion plan:")
print(adjusted_plan)

In [ ]:
def improve_energy_and_fiber(menu, candidate_df, constraints):
    menu = menu.copy()

    energy_total = pd.to_numeric(menu["energy_kcal_per_portion"], errors="coerce").sum()
    fiber_total = pd.to_numeric(menu["fiber_g_per_portion"], errors="coerce").sum()

    energy_min = constraints["energy_target"] * 0.90
    fiber_min = constraints.get("fiber_min_g", None)
    sodium_max = constraints.get("sodium_max_mg", None)

    need_energy = energy_total < energy_min
    need_fiber = fiber_min is not None and fiber_total < fiber_min

    if not need_energy and not need_fiber:
        return menu

    add_candidates = candidate_df[
        candidate_df["category_code"].isin(["S", "B", "LN", "MP"])
    ].copy()

    if sodium_max is not None:
        add_candidates = add_candidates[
            add_candidates["sodium_mg_per_portion"].fillna(0) <= 400
        ]

    add_candidates["fiber_score"] = (
        add_candidates["fiber_g_per_portion"].fillna(0) * 3
        + add_candidates["energy_kcal_per_portion"].fillna(0) * 0.02
        - add_candidates["sodium_mg_per_portion"].fillna(0) * 0.01
    )

    add_candidates = add_candidates.sort_values("fiber_score", ascending=False)

    max_extra_foods = 3
    original_len = len(menu)

    for _, row in add_candidates.head(10).iterrows():
        new_row = pd.DataFrame([row])
        menu = pd.concat([menu, new_row], ignore_index=True)

        # Force numeric after concat
        for col in [
            "energy_kcal_per_portion",
            "fiber_g_per_portion",
            "sodium_mg_per_portion"
        ]:
            if col in menu.columns:
                menu[col] = pd.to_numeric(menu[col], errors="coerce")

        energy_total = menu["energy_kcal_per_portion"].sum()
        fiber_total = menu["fiber_g_per_portion"].sum()
        sodium_total = menu["sodium_mg_per_portion"].sum()

        energy_ok = energy_total >= energy_min
        fiber_ok = True if fiber_min is None else fiber_total >= fiber_min
        sodium_ok = True if sodium_max is None else sodium_total <= sodium_max

        if energy_ok and fiber_ok and sodium_ok:
            break

        if len(menu) >= original_len + max_extra_foods:
            break

    return menu

In [ ]:
daily_menu_improved = improve_energy_and_fiber(
    daily_menu,
    candidate_for_patient,
    constraints
)

nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

existing_nutrient_cols = [
    col for col in nutrient_cols if col in daily_menu_improved.columns
]

# IMPORTANT: convert back to numeric after concat
for col in existing_nutrient_cols:
    daily_menu_improved[col] = pd.to_numeric(
        daily_menu_improved[col],
        errors="coerce"
    )

daily_totals_improved = daily_menu_improved[existing_nutrient_cols].sum()

menu_evaluation_improved = evaluate_menu(
    daily_totals_improved,
    constraints
)

display(
    daily_menu_improved[[
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]]
)

display(
    daily_totals_improved
    .reset_index()
    .rename(columns={"index": "nutrient", 0: "total"})
)

display(menu_evaluation_improved)

In [ ]:
display(
    daily_menu_improved[[
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "protein_g_per_portion",
        "fat_g_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "potassium_mg_per_portion",
        "calcium_mg_per_portion",
        "fiber_g_per_portion"
    ]]
)

In [ ]:
daily_menu_improved.to_csv("rule_based_menu_dm_ht_obesity.csv", index=False)
menu_evaluation_improved.to_csv("rule_based_menu_evaluation_dm_ht_obesity.csv", index=False)

daily_totals_improved.reset_index().rename(
    columns={"index": "nutrient", 0: "total"}
).to_csv("rule_based_menu_nutrient_total_dm_ht_obesity.csv", index=False)

print("Saved:")
print("- rule_based_menu_dm_ht_obesity.csv")
print("- rule_based_menu_evaluation_dm_ht_obesity.csv")
print("- rule_based_menu_nutrient_total_dm_ht_obesity.csv")